<a href="https://colab.research.google.com/github/AdebanjiAdelowo/Image_denoising_using_ResNet/blob/main/final_image_denoising.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%tensorflow_version 2.x
import tensorflow as tf

# Enable memory growth so the GPU allocates only what it needs
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# Mixed precision: use float16 on GPU, float32 for master weights
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print("Compute dtype:", mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", mixed_precision.global_policy().variable_dtype)

device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    raise SystemError('GPU device not found')
print('Found GPU at:', device_name)

In [ ]:
import numpy as np
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import mixed_precision
from tensorflow.keras.layers import (
    Input, Conv2D, Activation, Add, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from skimage import color
from skimage.metrics import structural_similarity as ssim_fn

In [ ]:
# LFW dataset — 13,000 face images at 250×250
# https://www.tensorflow.org/datasets/catalog/lfw
dataset, info = tfds.load("lfw", split="train", shuffle_files=True, with_info=True)
tfds.show_examples(dataset, info)  # new tfds API: ds first, info second
plt.show()

In [ ]:
def add_gaussian_noise(image, sigma):
    """Add zero-mean Gaussian noise with standard deviation sigma."""
    noise = np.random.normal(0, sigma, image.shape)
    return np.clip(image + noise, 0.0, 1.0).astype(np.float32)

In [ ]:
def add_salt_and_pepper_noise(image, frequency):
    """Replace each pixel independently with probability `frequency`."""
    noise = np.random.uniform(0, 1, image.shape)
    noisy = image.copy()
    noisy[noise <= frequency / 2] = 0.0
    noisy[1 - noise <= frequency / 2] = 1.0
    return noisy.astype(np.float32)

In [ ]:
# Take 5 000 images, convert to float32 grayscale [0, 1]
ds = dataset.shuffle(7000).take(5000)
images_clean = np.array([data["image"] for data in tfds.as_numpy(ds)])

# RGB → grayscale, add channel dim
images_clean = color.rgb2gray(images_clean).astype(np.float32)
images_clean = images_clean[:, :, :, np.newaxis]
print("Clean images shape:", images_clean.shape)

In [ ]:
SIGMA = 0.09
images_noisy = add_gaussian_noise(images_clean, SIGMA)

In [ ]:
train_noisy = images_noisy[:3000]
train_clean = images_clean[:3000]
test_noisy  = images_noisy[3000:]
test_clean  = images_clean[3000:]

print(f"Train: {train_clean.shape}, Test: {test_clean.shape}")

In [ ]:
def PSNR(orig, pred):
    """Peak Signal-to-Noise Ratio (dB). Higher is better."""
    mse = np.mean((orig.astype(np.float64) - pred.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    return 10.0 * np.log10(1.0 / mse)

def SSIM(orig, pred):
    """Structural Similarity Index. Range [−1, 1]; higher is better."""
    return ssim_fn(
        orig[:, :, 0].astype(np.float64),
        pred[:, :, 0].astype(np.float64),
        data_range=1.0
    )

In [ ]:
idx = 0
psnr_noisy = PSNR(test_clean[idx], test_noisy[idx])
ssim_noisy = SSIM(test_clean[idx], test_noisy[idx])

plt.figure(figsize=(10, 4))

plt.subplot(121)
plt.imshow(test_clean[idx, :, :, 0], cmap="binary_r")
plt.title("Original")
plt.axis('off')

plt.subplot(122)
plt.imshow(test_noisy[idx, :, :, 0], cmap="binary_r")
plt.title(f"Gaussian noise  PSNR={psnr_noisy:.1f} dB  SSIM={ssim_noisy:.3f}")
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def res_block(x, num_channels):
    """Residual block: Conv→BN→ReLU→Conv→BN → add skip → ReLU."""
    shortcut = x
    x = Conv2D(num_channels, (3, 3), padding='same',
                kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(num_channels, (3, 3), padding='same',
                kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Add()([shortcut, x])
    x = Activation('relu')(x)
    return x


def build_model(height, width, num_channels=64, num_res_blocks=5):
    inp = Input(shape=(height, width, 1))
    x   = Conv2D(num_channels, (3, 3), padding='same',
                 kernel_initializer='he_normal')(inp)
    x   = Activation('relu')(x)

    for _ in range(num_res_blocks):
        x = res_block(x, num_channels)

    # Final 1×1 projection back to 1 channel; cast to float32 for loss stability
    x   = Conv2D(1, (3, 3), padding='same',
                 kernel_initializer='he_normal', dtype='float32')(x)
    out = Add(dtype='float32')([inp, x])

    return Model(inputs=inp, outputs=out)


model = build_model(250, 250, num_channels=64, num_res_blocks=5)
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=6, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        'best_denoiser.keras', monitor='val_loss',
        save_best_only=True, verbose=1
    ),
]

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mean_squared_error',
    metrics=['mae'],
)

In [ ]:
history = model.fit(
    train_noisy, train_clean,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

epochs_ran = range(1, len(history.history['loss']) + 1)

axes[0].plot(epochs_ran, history.history['loss'],     'b-',  label='Train MSE')
axes[0].plot(epochs_ran, history.history['val_loss'], 'b--', label='Val MSE')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(epochs_ran, history.history['mae'],     'r-',  label='Train MAE')
axes[1].plot(epochs_ran, history.history['val_mae'], 'r--', label='Val MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Mean Absolute Error')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
loss, mae = model.evaluate(test_noisy, test_clean, verbose=1)
print(f"\nTest MSE: {loss:.6f}  |  Test MAE: {mae:.6f}")

In [ ]:
test_denoised = model.predict(test_noisy, batch_size=32, verbose=1)
# Clip predictions to valid range
test_denoised = np.clip(test_denoised, 0.0, 1.0).astype(np.float32)

In [ ]:
nimg = 5
plt.figure(figsize=(12, 3.5 * nimg))

for i in range(nimg):
    idx = np.random.randint(0, test_clean.shape[0])

    orig   = test_clean[idx]
    noisy  = test_noisy[idx]
    denoised = test_denoised[idx]

    psnr_n = PSNR(orig, noisy);    ssim_n = SSIM(orig, noisy)
    psnr_d = PSNR(orig, denoised); ssim_d = SSIM(orig, denoised)

    plt.subplot(nimg, 3, 3*i + 1)
    plt.imshow(orig[:, :, 0], cmap='binary_r')
    plt.axis('off')
    if i == 0: plt.title("Original")

    plt.subplot(nimg, 3, 3*i + 2)
    plt.imshow(noisy[:, :, 0], cmap='binary_r')
    plt.annotate(f"PSNR {psnr_n:.1f} dB\nSSIM {ssim_n:.3f}",
                 (0.98, 0.02), xycoords='axes fraction',
                 va='bottom', ha='right', color='red', fontsize=9)
    plt.axis('off')
    if i == 0: plt.title("Noisy (Gaussian)")

    plt.subplot(nimg, 3, 3*i + 3)
    plt.imshow(denoised[:, :, 0], cmap='binary_r')
    plt.annotate(f"PSNR {psnr_d:.1f} dB\nSSIM {ssim_d:.3f}",
                 (0.98, 0.02), xycoords='axes fraction',
                 va='bottom', ha='right', color='green', fontsize=9)
    plt.axis('off')
    if i == 0: plt.title("Denoised (ResNet)")

plt.tight_layout()
plt.savefig('denoised_results_gaussian.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Apply the trained model to salt-and-pepper noise (zero-shot generalisation test)
FREQ = 0.05
test_sp_noisy   = np.array([add_salt_and_pepper_noise(img, FREQ) for img in test_clean])
test_sp_denoised = np.clip(model.predict(test_sp_noisy, batch_size=32, verbose=1), 0, 1)

nimg = 5
plt.figure(figsize=(12, 3.5 * nimg))

for i in range(nimg):
    idx = np.random.randint(0, test_clean.shape[0])

    orig     = test_clean[idx]
    noisy    = test_sp_noisy[idx]
    denoised = test_sp_denoised[idx]

    psnr_n = PSNR(orig, noisy);    ssim_n = SSIM(orig, noisy)
    psnr_d = PSNR(orig, denoised); ssim_d = SSIM(orig, denoised)

    plt.subplot(nimg, 3, 3*i + 1)
    plt.imshow(orig[:, :, 0], cmap='binary_r')
    plt.axis('off')
    if i == 0: plt.title("Original")

    plt.subplot(nimg, 3, 3*i + 2)
    plt.imshow(noisy[:, :, 0], cmap='binary_r')
    plt.annotate(f"PSNR {psnr_n:.1f} dB\nSSIM {ssim_n:.3f}",
                 (0.98, 0.02), xycoords='axes fraction',
                 va='bottom', ha='right', color='red', fontsize=9)
    plt.axis('off')
    if i == 0: plt.title("Noisy (Salt & Pepper)")

    plt.subplot(nimg, 3, 3*i + 3)
    plt.imshow(denoised[:, :, 0], cmap='binary_r')
    plt.annotate(f"PSNR {psnr_d:.1f} dB\nSSIM {ssim_d:.3f}",
                 (0.98, 0.02), xycoords='axes fraction',
                 va='bottom', ha='right', color='green', fontsize=9)
    plt.axis('off')
    if i == 0: plt.title("Denoised (ResNet)")

plt.tight_layout()
plt.savefig('denoised_results_saltpepper.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
model.save('denoiser_resnet.keras')
print("Model saved to denoiser_resnet.keras")